In [1]:
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
import xgboost as xgb
from xgboost import XGBRegressor

In [2]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.analytic import ProbabilityOfImprovement
import copy

In [3]:
loaded_model = xgb.XGBRegressor()
loaded_model.load_model("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelXG/ModelXG.json")

In [4]:
def SurrogateModelOfReality(n_ci, n_it):
    y_pred = loaded_model.predict(np.array([[n_ci],[n_it]]).T)[0]
    return np.float64(y_pred)

In [5]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [6]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0, 1])),
        RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0, 1])),
        RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0, 1])),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": ProbabilityOfImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    # Quasirandom Sampling Exercise
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler.three.QuasirandomSampler3D_func(8,Parameters_lis).T

    for array in X:
        s1 = array[0]
        s2 = array[1]
        b1 = array[2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        my_parameters = {"s1": s1, "s2": s2, "b1": b1}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(n_ci,n_it)})

    for _ in range(7):
        IterationClient = copy.deepcopy(client)
        IterationTrials = {}
        for __ in range(3):
            SampleTrial = IterationClient.get_next_trials(max_trials=1)
            for trial_index, parameters in SampleTrial.items():
                IterationTrials[trial_index]=parameters
                s1 = parameters["s1"]
                s2 = parameters["s2"]
                b1 = parameters["b1"]
                result = IterationClient.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
                raw_data = {metric_name: result}
                IterationClient.complete_trial(trial_index=trial_index, raw_data=raw_data)
        for trial_index, parameters in IterationTrials.items():
            client.attach_trial(parameters=parameters)
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]
            n_ci = PredictorsToCaStoichs(s1,b1)
            n_it = PredictorsToIaStoichs(s2,b1)
            result = SurrogateModelOfReality(n_ci,n_it)
            raw_data = {metric_name: result}
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)

    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(np.array(client.summarize().t1).tolist()[0:27]))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
13.99153995513916

Trial 1 =========================================
17.348304748535156

Trial 2 =========================================
17.05820083618164

Trial 3 =========================================
13.955521583557129

Trial 4 =========================================
17.18518829345703

Trial 5 =========================================
13.758429527282715

Trial 6 =========================================
15.329530715942383

Trial 7 =========================================
16.268138885498047

Trial 8 =========================================
15.394299507141113

Trial 9 =========================================
13.955521583557129



/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/botorch/fit.py:215: OptimizationWarning: `scipy_minimize` terminated with status OptimizationStatus.FAILURE, displaying original message from `scipy.optimize.minimize`: ABNORMAL: 
  result = optimizer(mll, closure=closure, **optimizer_kwargs)


Trial 10 =========================================
15.7681884765625

Trial 11 =========================================
15.62542724609375

Trial 12 =========================================
14.314135551452637

Trial 13 =========================================
13.955521583557129



/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/botorch/optim/optimize.py:677: RuntimeWarning: Optimization failed in `gen_candidates_scipy` with the following warning(s):
[OptimizationWarning('Optimization failed within `scipy.optimize.minimize` with status 2 and message ABNORMAL: .')]
Trying again with a new set of initial conditions.
  return _optimize_acqf_batch(opt_inputs=opt_inputs)


Trial 14 =========================================
16.545217514038086

Trial 15 =========================================
14.312739372253418

Trial 16 =========================================
16.545217514038086

Trial 17 =========================================
13.495748519897461

Trial 18 =========================================
16.268138885498047

Trial 19 =========================================
16.349613189697266

Trial 20 =========================================
17.13752555847168

Trial 21 =========================================
17.348304748535156

Trial 22 =========================================
16.545217514038086

Trial 23 =========================================
15.329530715942383

Trial 24 =========================================
15.68480396270752

Trial 25 =========================================
14.314135551452637

Trial 26 =========================================
15.68480396270752

Trial 27 =========================================
14.690720558166504

Trial 28 

/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/botorch/fit.py:215: OptimizationWarning: `scipy_minimize` terminated with status OptimizationStatus.FAILURE, displaying original message from `scipy.optimize.minimize`: ABNORMAL: 
  result = optimizer(mll, closure=closure, **optimizer_kwargs)


Trial 33 =========================================
13.955521583557129

Trial 34 =========================================
13.090934753417969

Trial 35 =========================================
13.893892288208008

Trial 36 =========================================
13.522225379943848

Trial 37 =========================================
13.893892288208008

Trial 38 =========================================
13.955521583557129

Trial 39 =========================================
17.348304748535156

Trial 40 =========================================
13.885415077209473

Trial 41 =========================================
13.893892288208008

Trial 42 =========================================
16.036523818969727

Trial 43 =========================================
16.6074275970459

Trial 44 =========================================
15.112234115600586

Trial 45 =========================================
17.348304748535156

Trial 46 =========================================
16.35771369934082

Trial 47 

/Users/thomasdodd/miniconda3/envs/ax1xgb_env/lib/python3.12/site-packages/botorch/optim/optimize.py:677: RuntimeWarning: Optimization failed in `gen_candidates_scipy` with the following warning(s):
[OptimizationWarning('Optimization failed within `scipy.optimize.minimize` with status 2 and message ABNORMAL: .')]
Trying again with a new set of initial conditions.
  return _optimize_acqf_batch(opt_inputs=opt_inputs)


Trial 93 =========================================
15.637456893920898

Trial 94 =========================================
16.545217514038086

Trial 95 =========================================
17.05820083618164

Trial 96 =========================================
17.05820083618164

Trial 97 =========================================
13.758429527282715

Trial 98 =========================================
13.893892288208008

Trial 99 =========================================
13.556435585021973

[13.99153996 17.34830475 17.05820084 13.95552158 17.18518829 13.75842953
 15.32953072 16.26813889 15.39429951 13.95552158 15.76818848 15.62542725
 14.31413555 13.95552158 16.54521751 14.31273937 16.54521751 13.49574852
 16.26813889 16.34961319 17.13752556 17.34830475 16.54521751 15.32953072
 15.68480396 14.31413555 15.68480396 14.69072056 14.37495804 14.00028896
 14.21930599 13.95552158 13.89389229 13.95552158 13.09093475 13.89389229
 13.52222538 13.89389229 13.95552158 17.34830475 13.88541508 13.893

In [7]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.626216888427734
Avg = 15.316152477264405
Std = 1.2537918920440723


In [8]:
print(y_max_arr.tolist())

[13.99153995513916, 17.348304748535156, 17.05820083618164, 13.955521583557129, 17.18518829345703, 13.758429527282715, 15.329530715942383, 16.268138885498047, 15.394299507141113, 13.955521583557129, 15.7681884765625, 15.62542724609375, 14.314135551452637, 13.955521583557129, 16.545217514038086, 14.312739372253418, 16.545217514038086, 13.495748519897461, 16.268138885498047, 16.349613189697266, 17.13752555847168, 17.348304748535156, 16.545217514038086, 15.329530715942383, 15.68480396270752, 14.314135551452637, 15.68480396270752, 14.690720558166504, 14.374958038330078, 14.000288963317871, 14.219305992126465, 13.955521583557129, 13.893892288208008, 13.955521583557129, 13.090934753417969, 13.893892288208008, 13.522225379943848, 13.893892288208008, 13.955521583557129, 17.348304748535156, 13.885415077209473, 13.893892288208008, 16.036523818969727, 16.6074275970459, 15.112234115600586, 17.348304748535156, 16.35771369934082, 14.825654983520508, 15.394299507141113, 14.017738342285156, 16.39373588

In [9]:
# filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/SequentialTestswGPModel/DataGenerated/normal_PI_9_27_3.pkl"
# latestdf = pd.DataFrame(y_max_arr)
# pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)

In [10]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelXG/DataGenerated/normal_PI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [11]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelXG/DataGenerated/normal_PI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    16.545218
1    13.893892
2    13.495749
3    13.885415
4    14.017738
..         ...
995  17.058201
996  17.058201
997  13.758430
998  13.893892
999  13.556436

[1000 rows x 1 columns]
